# HoloSyn Visual Interface (Gradio) — Integrates Your Distilled TorchScript Model

This Colab notebook builds a **visual UI** to:
- Load your distilled model: `/mnt/data/student_distilled_heads_hf.torchscript.pt`
- Load normalization: `/mnt/data/student_norm_hf.json`
- Optionally load your archive: `/mnt/data/Archive.zip` and browse files
- Compute modality-specific features (text/audio/image/video/haptics)
- Run inference → **valence/arousal/calm/trust**
- Visualize:
  - meters + time series
  - optional **two-peer synchrony** (A/B)
- Export a JSON session log

**Privacy note:** Everything runs locally in the notebook runtime.

---

## Inputs expected
- `/mnt/data/student_distilled_heads_hf.torchscript.pt`
- `/mnt/data/student_norm_hf.json`
- Optional: `/mnt/data/Archive.zip`


In [1]:
#@title 0) Install deps
#@title 0) Install deps
!pip -q install -U gradio numpy "pandas==2.2.2" pillow opencv-python soundfile librosa ffmpeg-python sentence-transformers transformers
!pip -q install -U torch torchvision torchaudio pyarrow
import os, json, time, zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import librosa
import soundfile as sf
import cv2

import gradio as gr
print("✅ Installed")

✅ Installed


In [2]:
#@title 1) Paths + load model
MODEL_PATH = "/content/student_distilled_heads_hf.torchscript.pt"
NORM_PATH  = "/content/student_norm_hf.json"
ARCHIVE_ZIP = "/content/Archive.zip"

assert os.path.exists(MODEL_PATH), f"Missing model: {MODEL_PATH}"
assert os.path.exists(NORM_PATH), f"Missing norm: {NORM_PATH}"

model = torch.jit.load(MODEL_PATH)
model.eval()

with open(NORM_PATH, "r", encoding="utf-8") as f:
    norm = json.load(f)

NUMERIC_COLS = norm["numeric_cols"]
MU = np.array(norm["mu"], dtype=np.float32)
SD = np.array(norm["sd"], dtype=np.float32)

print("✅ Model loaded")
print("Feature dims:", len(NUMERIC_COLS))

✅ Model loaded
Feature dims: 789


In [3]:
#@title 2) Optional: extract archive and index files
EXTRACT_DIR = "/content/archive_extracted_ui"
Path(EXTRACT_DIR).mkdir(parents=True, exist_ok=True)

AUDIO_EXT = {".wav",".mp3",".m4a",".flac",".ogg",".aac"}
VIDEO_EXT = {".mp4",".mov",".mkv",".webm",".avi"}
IMAGE_EXT = {".png",".jpg",".jpeg",".webp",".bmp"}
TEXT_EXT  = {".txt",".md",".json",".csv",".tsv"}

def extract_archive_if_present():
    if os.path.exists(ARCHIVE_ZIP):
        with zipfile.ZipFile(ARCHIVE_ZIP, "r") as z:
            z.extractall(EXTRACT_DIR)
        return True
    return False

def walk_files(root):
    out=[]
    for p in Path(root).rglob("*"):
        if p.is_file():
            out.append(str(p))
    return out

def bucket(path):
    ext = Path(path).suffix.lower()
    low = path.lower()
    if ext in AUDIO_EXT: return "audio"
    if ext in VIDEO_EXT: return "video"
    if ext in IMAGE_EXT: return "image"
    if ext in TEXT_EXT:
        if ext in {".json",".csv",".tsv"} and any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
            return "haptics"
        return "text"
    if any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
        return "haptics"
    return "other"

HAS_ARCHIVE = extract_archive_if_present()
INDEX = {"audio":[], "video":[], "image":[], "text":[], "haptics":[], "other":[]}
if HAS_ARCHIVE:
    for f in walk_files(EXTRACT_DIR):
        INDEX[bucket(f)].append(f)

print("Archive present:", HAS_ARCHIVE)
if HAS_ARCHIVE:
    for k in ["audio","video","image","text","haptics","other"]:
        print(k, len(INDEX[k]))


Archive present: True
audio 49
video 52
image 407
text 49
haptics 97
other 25


In [4]:
#@title 3) Feature extraction (must align with training feature schema)
def load_text(path, max_chars=12000):
    return open(path, "r", encoding="utf-8", errors="ignore").read()[:max_chars]

def load_haptics_any(path):
    ext = Path(path).suffix.lower()
    if ext == ".json":
        try:
            return json.load(open(path, "r", encoding="utf-8", errors="ignore"))
        except Exception:
            return {"raw": load_text(path)}
    if ext in {".csv",".tsv"}:
        sep = "," if ext==".csv" else "\t"
        try:
            df = pd.read_csv(path, sep=sep)
            return {"columns": list(df.columns), "head": df.head(200).to_dict(orient="list")}
        except Exception:
            return {"raw": load_text(path)}
    return {"raw": load_text(path)}

def featurize_text(s):
    return {
        "txt_len": float(len(s)),
        "txt_lines": float(s.count("\n")+1),
        "txt_exclaim": float(s.count("!")),
        "txt_question": float(s.count("?")),
        "txt_caps_ratio": float(sum(c.isupper() for c in s)/max(1,len(s))),
    }

def featurize_haptics(h):
    raw = json.dumps(h)[:20000].lower()
    return {
        "hapt_len": float(len(raw)),
        "hapt_has_intensity": 1.0 if "intensity" in raw else 0.0,
        "hapt_has_freq": 1.0 if ("hz" in raw or "freq" in raw) else 0.0,
    }

def audio_features(path, sr=16000, max_seconds=15):
    y, _ = librosa.load(path, sr=sr, mono=True, duration=max_seconds)
    if y.size < 10:
        return {"aud_rms":0.0,"aud_zcr":0.0,"aud_centroid":0.0,"aud_tempo":0.0}
    rms = float(np.mean(librosa.feature.rms(y=y)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    tempo = float(librosa.beat.tempo(onset_envelope=onset_env, sr=sr)[0]) if onset_env.size else 0.0
    return {"aud_rms":rms,"aud_zcr":zcr,"aud_centroid":centroid,"aud_tempo":tempo}

def image_quick_stats(path):
    im = Image.open(path).convert("RGB")
    arr = np.asarray(im).astype(np.float32)/255.0
    mean = arr.mean(axis=(0,1))
    std = arr.std(axis=(0,1))
    return {
        "img_w": float(arr.shape[1]), "img_h": float(arr.shape[0]),
        "img_mean_r": float(mean[0]), "img_mean_g": float(mean[1]), "img_mean_b": float(mean[2]),
        "img_std_r": float(std[0]), "img_std_g": float(std[1]), "img_std_b": float(std[2]),
    }

def sample_video_frames(video_path, every_n_frames=45, max_frames=4, target_size=224):
    cap = cv2.VideoCapture(video_path)
    frames=[]
    idx=0
    while cap.isOpened() and len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret: break
        if idx % every_n_frames == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h,w = frame.shape[:2]
            if max(h,w) > target_size:
                scale = target_size / max(h,w)
                frame = cv2.resize(frame, (int(w*scale), int(h*scale)))
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

def make_feature_vector(modality, path_or_text):
    # Return dict of features; missing features are 0.0.
    feats = {}
    preview = None

    if modality == "text":
        s = path_or_text if isinstance(path_or_text, str) and "\n" in path_or_text else load_text(path_or_text)
        feats.update(featurize_text(s))
        preview = s[:1000]
    elif modality == "haptics":
        h = load_haptics_any(path_or_text)
        feats.update(featurize_haptics(h))
        preview = json.dumps(h)[:1000]
    elif modality == "audio":
        feats.update(audio_features(path_or_text))
        preview = f"Audio file: {Path(path_or_text).name}"
    elif modality == "image":
        feats.update(image_quick_stats(path_or_text))
        preview = Image.open(path_or_text).convert("RGB")
    elif modality == "video":
        frames = sample_video_frames(path_or_text)
        feats["vid_n_frames"] = float(len(frames))
        preview = Image.fromarray(frames[0]).convert("RGB") if frames else None
    else:
        preview = f"Unsupported modality for: {path_or_text}"

    # Build full vector in the exact order expected by the student
    x = np.zeros((len(NUMERIC_COLS),), dtype=np.float32)
    for i, col in enumerate(NUMERIC_COLS):
        if col in feats:
            x[i] = np.float32(feats[col])
        else:
            # embeddings columns (clip_*, w2v_*) are not computed here; keep 0 unless you add embed models
            x[i] = 0.0
    return x, feats, preview

def predict_from_x(x):
    xn = (x - MU) / SD
    xt = torch.tensor(xn, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        y = model(xt).squeeze(0).cpu().numpy().astype(np.float32)
    # [valence, arousal, calm, trust]
    return y

In [5]:
#@title 4) Session logger utilities
SESSION_LOG = []

def log_step(label, modality, source, y, feats):
    rec = {
        "ts": time.time(),
        "label": label,
        "modality": modality,
        "source": source,
        "valence": float(y[0]),
        "arousal": float(y[1]),
        "calm": float(y[2]),
        "trust": float(y[3]),
        "features": {k: float(v) if isinstance(v,(int,float,np.floating)) else str(v) for k,v in feats.items()}
    }
    SESSION_LOG.append(rec)

def export_log():
    out = "/mnt/data/holosyn_session_log.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(SESSION_LOG, f, indent=2)
    return out

## 5) Gradio UI

Two panels:
- **Single input inference** (pick modality + source)
- **Two-peer synchrony**: run A & B and compute cosine similarity on `[valence, arousal, calm, trust]`


In [6]:
#@title 5) Build UI
def list_options(modality):
    if not HAS_ARCHIVE:
        return []
    return [p.replace(EXTRACT_DIR + "/", "") for p in INDEX.get(modality, [])][:500]

def resolve_path(rel_path):
    if rel_path is None or rel_path == "":
        return None
    return os.path.join(EXTRACT_DIR, rel_path)

def infer_one(modality, rel_file, free_text):
    if modality == "text" and (free_text and free_text.strip()):
        x, feats, preview = make_feature_vector("text", free_text)
        y = predict_from_x(x)
        log_step("single", "text", "free_text", y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats
    else:
        path = resolve_path(rel_file)
        if not path or not os.path.exists(path):
            return "Pick a file from the dropdown (or paste text).", 0,0,0,0, {}
        x, feats, preview = make_feature_vector(modality, path)
        y = predict_from_x(x)
        log_step("single", modality, rel_file, y, feats)
        # preview might be PIL Image
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats

def infer_pair(modalityA, fileA, textA, modalityB, fileB, textB):
    prevA, vA, aA, cA, tA, featsA = infer_one(modalityA, fileA, textA)
    prevB, vB, aB, cB, tB, featsB = infer_one(modalityB, fileB, textB)
    eA = np.array([vA,aA,cA,tA], dtype=np.float32)
    eB = np.array([vB,aB,cB,tB], dtype=np.float32)
    sync = float(np.dot(eA, eB) / (np.linalg.norm(eA)*np.linalg.norm(eB) + 1e-8))
    return prevA, prevB, sync

def do_export():
    path = export_log()
    return path

with gr.Blocks(title="HoloSyn UI") as demo:
    gr.Markdown("## HoloSyn Visual Interface (Local Distilled Model)")
    if not HAS_ARCHIVE:
        gr.Markdown("⚠️ `Archive.zip` not found. Upload it to `/mnt/data/Archive.zip` for file browsing. Text mode still works.")

    with gr.Tab("Single"):
        with gr.Row():
            modality = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality")
            file_dd = gr.Dropdown(choices=list_options("text"), label="File from Archive (optional)")
        free_text = gr.Textbox(lines=6, label="Text input (only used if modality=text and non-empty)")
        run_btn = gr.Button("Run inference")
        with gr.Row():
            preview = gr.Image(label="Preview (image/video frame) OR Text snippet", type="pil")
            preview_txt = gr.Textbox(label="Preview text (if not an image)", lines=8)
        with gr.Row():
            val = gr.Slider(0,1, step=0.001, label="Valence", interactive=False)
            aro = gr.Slider(0,1, step=0.001, label="Arousal", interactive=False)
            calm = gr.Slider(0,1, step=0.001, label="Calm", interactive=False)
            trust = gr.Slider(0,1, step=0.001, label="Trust", interactive=False)
        feats_json = gr.JSON(label="Extracted features used by student")

        def refresh_files(m):
            return gr.Dropdown.update(choices=list_options(m), value=None)
        modality.change(refresh_files, inputs=[modality], outputs=[file_dd])

        def render(preview_obj, v,a,c,t, feats):
            # If preview is an image, show it; else show text in preview_txt
            if isinstance(preview_obj, Image.Image):
                return preview_obj, "", v,a,c,t, feats
            else:
                # show placeholder image blank
                return None, str(preview_obj), v,a,c,t, feats

        run_btn.click(
            fn=lambda m,f,txt: infer_one(m,f,txt),
            inputs=[modality,file_dd,free_text],
            outputs=[preview_txt,val,aro,calm,trust,feats_json],
        ).then(
            fn=lambda prev_txt, v,a,c,t, feats: render(prev_txt, v,a,c,t, feats),
            inputs=[preview_txt,val,aro,calm,trust,feats_json],
            outputs=[preview,preview_txt,val,aro,calm,trust,feats_json]
        )

    with gr.Tab("Two-peer synchrony"):
        gr.Markdown("Run two inputs (A/B) and compute synchrony on the model outputs.")
        with gr.Row():
            modalityA = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality A")
            fileA = gr.Dropdown(choices=list_options("text"), label="File A")
        textA = gr.Textbox(lines=4, label="Text A (used if Modality A=text and non-empty)")
        with gr.Row():
            modalityB = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality B")
            fileB = gr.Dropdown(choices=list_options("text"), label="File B")
        textB = gr.Textbox(lines=4, label="Text B (used if Modality B=text and non-empty)")
        run_pair = gr.Button("Run pair + synchrony")
        with gr.Row():
            prevA = gr.Image(label="Preview A", type="pil")
            prevB = gr.Image(label="Preview B", type="pil")
        sync = gr.Slider(0,1, step=0.001, label="Synchrony (cosine similarity)", interactive=False)

        modalityA.change(lambda m: gr.Dropdown.update(choices=list_options(m), value=None), inputs=[modalityA], outputs=[fileA])
        modalityB.change(lambda m: gr.Dropdown.update(choices=list_options(m), value=None), inputs=[modalityB], outputs=[fileB])

        run_pair.click(
            fn=infer_pair,
            inputs=[modalityA,fileA,textA, modalityB,fileB,textB],
            outputs=[prevA,prevB,sync]
        )

    with gr.Tab("Export session log"):
        gr.Markdown("Exports all inference steps recorded so far to JSON.")
        export_btn = gr.Button("Export log")
        out_path = gr.Textbox(label="Saved to")
        export_btn.click(fn=do_export, inputs=[], outputs=[out_path])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://50c6bff44aa90dc61f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [7]:
#@title 5) Build UI
def list_options(modality):
    if not HAS_ARCHIVE:
        return []
    return [p.replace(EXTRACT_DIR + "/", "") for p in INDEX.get(modality, [])][:500]

def resolve_path(rel_path):
    if rel_path is None or rel_path == "":
        return None
    return os.path.join(EXTRACT_DIR, rel_path)

def infer_one(modality, rel_file, free_text):
    if modality == "text" and (free_text and free_text.strip()):
        x, feats, preview = make_feature_vector("text", free_text)
        y = predict_from_x(x)
        log_step("single", "text", "free_text", y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats
    else:
        path = resolve_path(rel_file)
        if not path or not os.path.exists(path):
            return "Pick a file from the dropdown (or paste text).", 0,0,0,0, {}
        x, feats, preview = make_feature_vector(modality, path)
        y = predict_from_x(x)
        log_step("single", modality, rel_file, y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats

def do_export():
    path = export_log()
    return path

with gr.Blocks(title="HoloSyn UI") as demo:
    gr.Markdown("## HoloSyn Visual Interface (Local Distilled Model)")
    if not HAS_ARCHIVE:
        gr.Markdown("⚠️ `Archive.zip` not found. Upload it to `/mnt/data/Archive.zip` for file browsing. Text mode still works.")

    with gr.Tab("Single"):
        with gr.Row():
            modality = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality")
            file_dd = gr.Dropdown(choices=list_options("text"), label="File from Archive (optional)")
        free_text = gr.Textbox(lines=6, label="Text input (only used if modality=text and non-empty)")
        run_btn = gr.Button("Run inference")

        with gr.Row():
            preview = gr.Image(label="Preview (image/video frame)", type="pil")
            preview_txt = gr.Textbox(label="Preview text (if not an image)", lines=8)

        with gr.Row():
            val = gr.Slider(0,1, step=0.001, label="Valence", interactive=False)
            aro = gr.Slider(0,1, step=0.001, label="Arousal", interactive=False)
            calm = gr.Slider(0,1, step=0.001, label="Calm", interactive=False)
            trust = gr.Slider(0,1, step=0.001, label="Trust", interactive=False)
        feats_json = gr.JSON(label="Extracted features used by student")

        def refresh_files(m):
            return gr.Dropdown.update(choices=list_options(m), value=None)
        modality.change(refresh_files, inputs=[modality], outputs=[file_dd])

        # --- FIX: Unified handler to safely route text vs images ---
        def process_single(m, f, txt):
            prev_obj, v, a, c, t, feats = infer_one(m, f, txt)
            if isinstance(prev_obj, Image.Image):
                return prev_obj, "", v, a, c, t, feats
            else:
                return None, str(prev_obj), v, a, c, t, feats

        run_btn.click(
            fn=process_single,
            inputs=[modality, file_dd, free_text],
            outputs=[preview, preview_txt, val, aro, calm, trust, feats_json]
        )

    with gr.Tab("Two-peer synchrony"):
        gr.Markdown("Run two inputs (A/B) and compute synchrony on the model outputs.")
        with gr.Row():
            modalityA = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality A")
            fileA = gr.Dropdown(choices=list_options("text"), label="File A")
        textA = gr.Textbox(lines=4, label="Text A (used if Modality A=text and non-empty)")

        with gr.Row():
            modalityB = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality B")
            fileB = gr.Dropdown(choices=list_options("text"), label="File B")
        textB = gr.Textbox(lines=4, label="Text B (used if Modality B=text and non-empty)")

        run_pair = gr.Button("Run pair + synchrony")

        # --- FIX: Provide both Image and Text preview blocks for safely rendering dynamic modalities ---
        with gr.Row():
            with gr.Column():
                prevA_img = gr.Image(label="Preview A (Image/Video)", type="pil")
                prevA_txt = gr.Textbox(label="Preview A (Text/Other)", lines=4)
            with gr.Column():
                prevB_img = gr.Image(label="Preview B (Image/Video)", type="pil")
                prevB_txt = gr.Textbox(label="Preview B (Text/Other)", lines=4)

        sync = gr.Slider(0,1, step=0.001, label="Synchrony (cosine similarity)", interactive=False)

        modalityA.change(lambda m: gr.Dropdown.update(choices=list_options(m), value=None), inputs=[modalityA], outputs=[fileA])
        modalityB.change(lambda m: gr.Dropdown.update(choices=list_options(m), value=None), inputs=[modalityB], outputs=[fileB])

        # --- FIX: Unified handler to safely route pairs ---
        def process_pair(mA, fA, txtA, mB, fB, txtB):
            prevA, vA, aA, cA, tA, featsA = infer_one(mA, fA, txtA)
            prevB, vB, aB, cB, tB, featsB = infer_one(mB, fB, txtB)

            eA = np.array([vA,aA,cA,tA], dtype=np.float32)
            eB = np.array([vB,aB,cB,tB], dtype=np.float32)
            sync_val = float(np.dot(eA, eB) / (np.linalg.norm(eA)*np.linalg.norm(eB) + 1e-8))

            outA_img = prevA if isinstance(prevA, Image.Image) else None
            outA_txt = "" if isinstance(prevA, Image.Image) else str(prevA)
            outB_img = prevB if isinstance(prevB, Image.Image) else None
            outB_txt = "" if isinstance(prevB, Image.Image) else str(prevB)

            return outA_img, outA_txt, outB_img, outB_txt, sync_val

        run_pair.click(
            fn=process_pair,
            inputs=[modalityA, fileA, textA, modalityB, fileB, textB],
            outputs=[prevA_img, prevA_txt, prevB_img, prevB_txt, sync]
        )

    with gr.Tab("Export session log"):
        gr.Markdown("Exports all inference steps recorded so far to JSON.")
        export_btn = gr.Button("Export log")
        out_path = gr.Textbox(label="Saved to")
        export_btn.click(fn=do_export, inputs=[], outputs=[out_path])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://54628bd43af5f1a363.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
#@title 1) Imports & Load Model
import os, json, time, zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import torch
import librosa
import soundfile as sf
import cv2
import gradio as gr
from sentence_transformers import SentenceTransformer

MODEL_PATH = "./student_distilled_heads_hf.torchscript.pt"
NORM_PATH  = "./student_norm_hf.json"
ARCHIVE_ZIP = "./Archive.zip"

print("Loading text embedding model (this may take a moment)...")
embedder = SentenceTransformer('all-mpnet-base-v2')

assert os.path.exists(MODEL_PATH), f"Missing model: {MODEL_PATH}"
assert os.path.exists(NORM_PATH), f"Missing norm: {NORM_PATH}"

model = torch.jit.load(MODEL_PATH)
model.eval()

with open(NORM_PATH, "r", encoding="utf-8") as f:
    norm = json.load(f)

NUMERIC_COLS = norm["numeric_cols"]
MU = np.array(norm["mu"], dtype=np.float32)
SD = np.array(norm["sd"], dtype=np.float32)

print("✅ Model and Norm loaded")
print("Feature dims:", len(NUMERIC_COLS))

Loading text embedding model (this may take a moment)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model and Norm loaded
Feature dims: 789


In [9]:
#@title 1) Imports & Load Model
import os, json, time, zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import torch
import librosa
import soundfile as sf
import cv2
import gradio as gr
from sentence_transformers import SentenceTransformer
import transformers
import huggingface_hub

print(f"Transformers version: {transformers.__version__}")
print(f"Hugging Face Hub version: {huggingface_hub.__version__}")

MODEL_PATH = "./student_distilled_heads_hf.torchscript.pt"
NORM_PATH  = "./student_norm_hf.json"
ARCHIVE_ZIP = "./Archive.zip"

print("Loading text embedding model (this may take a moment)...")
embedder = SentenceTransformer('all-mpnet-base-v2')

assert os.path.exists(MODEL_PATH), f"Missing model: {MODEL_PATH}"
assert os.path.exists(NORM_PATH), f"Missing norm: {NORM_PATH}"

model = torch.jit.load(MODEL_PATH)
model.eval()

with open(NORM_PATH, "r", encoding="utf-8") as f:
    norm = json.load(f)

NUMERIC_COLS = norm["numeric_cols"]
MU = np.array(norm["mu"], dtype=np.float32)
SD = np.array(norm["sd"], dtype=np.float32)

print("✅ Model and Norm loaded")
print("Feature dims:", len(NUMERIC_COLS))

Transformers version: 5.2.0
Hugging Face Hub version: 1.5.0
Loading text embedding model (this may take a moment)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model and Norm loaded
Feature dims: 789


In [10]:
#@title 2) Optional: extract archive and index files
EXTRACT_DIR = "/content/archive_extracted_ui"
Path(EXTRACT_DIR).mkdir(parents=True, exist_ok=True)

AUDIO_EXT = {".wav",".mp3",".m4a",".flac",".ogg",".aac"}
VIDEO_EXT = {".mp4",".mov",".mkv",".webm",".avi"}
IMAGE_EXT = {".png",".jpg",".jpeg",".webp",".bmp"}
TEXT_EXT  = {".txt",".md",".json",".csv",".tsv"}

def extract_archive_if_present():
    if os.path.exists(ARCHIVE_ZIP):
        with zipfile.ZipFile(ARCHIVE_ZIP, "r") as z:
            z.extractall(EXTRACT_DIR)
        return True
    return False

def walk_files(root):
    out=[]
    for p in Path(root).rglob("*"):
        if p.is_file():
            out.append(str(p))
    return out

def bucket(path):
    ext = Path(path).suffix.lower()
    low = path.lower()
    if ext in AUDIO_EXT: return "audio"
    if ext in VIDEO_EXT: return "video"
    if ext in IMAGE_EXT: return "image"
    if ext in TEXT_EXT:
        if ext in {".json",".csv",".tsv"} and any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
            return "haptics"
        return "text"
    if any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
        return "haptics"
    return "other"

HAS_ARCHIVE = extract_archive_if_present()
INDEX = {"audio":[], "video":[], "image":[], "text":[], "haptics":[], "other":[]}
if HAS_ARCHIVE:
    for f in walk_files(EXTRACT_DIR):
        INDEX[bucket(f)].append(f)

print("Archive present:", HAS_ARCHIVE)
if HAS_ARCHIVE:
    for k in ["audio","video","image","text","haptics","other"]:
        print(k, len(INDEX[k]))

Archive present: True
audio 49
video 52
image 407
text 49
haptics 97
other 25


In [11]:
#@title 3) Feature extraction
def load_text(path, max_chars=12000):
    return open(path, "r", encoding="utf-8", errors="ignore").read()[:max_chars]

def load_haptics_any(path):
    ext = Path(path).suffix.lower()
    if ext == ".json":
        try:
            return json.load(open(path, "r", encoding="utf-8", errors="ignore"))
        except Exception:
            return {"raw": load_text(path)}
    if ext in {".csv",".tsv"}:
        sep = "," if ext==".csv" else "\t"
        try:
            df = pd.read_csv(path, sep=sep)
            return {"columns": list(df.columns), "head": df.head(200).to_dict(orient="list")}
        except Exception:
            return {"raw": load_text(path)}
    return {"raw": load_text(path)}

def featurize_text(s):
    return {
        "txt_len": float(len(s)),
        "txt_lines": float(s.count("\n")+1),
        "txt_exclaim": float(s.count("!")),
        "txt_question": float(s.count("?")),
        "txt_caps_ratio": float(sum(c.isupper() for c in s)/max(1,len(s))),
    }

def featurize_haptics(h):
    raw = json.dumps(h)[:20000].lower()
    return {
        "hapt_len": float(len(raw)),
        "hapt_has_intensity": 1.0 if "intensity" in raw else 0.0,
        "hapt_has_freq": 1.0 if ("hz" in raw or "freq" in raw) else 0.0,
    }

def audio_features(path, sr=16000, max_seconds=15):
    y, _ = librosa.load(path, sr=sr, mono=True, duration=max_seconds)
    if y.size < 10:
        return {"aud_rms":0.0,"aud_zcr":0.0,"aud_centroid":0.0,"aud_tempo":0.0}
    rms = float(np.mean(librosa.feature.rms(y=y)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    tempo = float(librosa.beat.tempo(onset_envelope=onset_env, sr=sr)[0]) if onset_env.size else 0.0
    return {"aud_rms":rms,"aud_zcr":zcr,"aud_centroid":centroid,"aud_tempo":tempo}

def image_quick_stats(path):
    im = Image.open(path).convert("RGB")
    arr = np.asarray(im).astype(np.float32)/255.0
    mean = arr.mean(axis=(0,1))
    std = arr.std(axis=(0,1))
    return {
        "img_w": float(arr.shape[1]), "img_h": float(arr.shape[0]),
        "img_mean_r": float(mean[0]), "img_mean_g": float(mean[1]), "img_mean_b": float(mean[2]),
        "img_std_r": float(std[0]), "img_std_g": float(std[1]), "img_std_b": float(std[2]),
    }

def sample_video_frames(video_path, every_n_frames=45, max_frames=4, target_size=224):
    cap = cv2.VideoCapture(video_path)
    frames=[]
    idx=0
    while cap.isOpened() and len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret: break
        if idx % every_n_frames == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h,w = frame.shape[:2]
            if max(h,w) > target_size:
                scale = target_size / max(h,w)
                frame = cv2.resize(frame, (int(w*scale), int(h*scale)))
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

def make_feature_vector(modality, path_or_text):
    feats = {}
    preview = None

    if modality == "text":
        s = path_or_text if isinstance(path_or_text, str) and "\n" in path_or_text else load_text(path_or_text)
        feats.update(featurize_text(s))
        # Compute the 768-d embedding
        emb = embedder.encode(s)
        for i in range(768):
            feats[f"w2v_{i}"] = float(emb[i])
        preview = s[:1000]

    elif modality == "haptics":
        h = load_haptics_any(path_or_text)
        feats.update(featurize_haptics(h))
        preview = json.dumps(h)[:1000]
    elif modality == "audio":
        feats.update(audio_features(path_or_text))
        preview = f"Audio file: {Path(path_or_text).name}"
    elif modality == "image":
        feats.update(image_quick_stats(path_or_text))
        preview = Image.open(path_or_text).convert("RGB")
    elif modality == "video":
        frames = sample_video_frames(path_or_text)
        feats["vid_n_frames"] = float(len(frames))
        preview = Image.fromarray(frames[0]).convert("RGB") if frames else None
    else:
        preview = f"Unsupported modality for: {path_or_text}"

    # Build full vector
    x = np.zeros((len(NUMERIC_COLS),), dtype=np.float32)
    for i, col in enumerate(NUMERIC_COLS):
        if col in feats:
            x[i] = np.float32(feats[col])
        else:
            x[i] = 0.0
    return x, feats, preview

def predict_from_x(x):
    xn = (x - MU) / SD
    xt = torch.tensor(xn, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        y = model(xt).squeeze(0).cpu().numpy().astype(np.float32)
    return y

In [12]:
#@title 4) Session logger utilities
SESSION_LOG = []

def log_step(label, modality, source, y, feats):
    rec = {
        "ts": time.time(),
        "label": label,
        "modality": modality,
        "source": source,
        "valence": float(y[0]),
        "arousal": float(y[1]),
        "calm": float(y[2]),
        "trust": float(y[3]),
        "features": {k: float(v) if isinstance(v,(int,float,np.floating)) else str(v) for k,v in feats.items()}
    }
    SESSION_LOG.append(rec)

def export_log():
    out = "/mnt/data/holosyn_session_log.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(SESSION_LOG, f, indent=2)
    return out

In [13]:
#@title 5) Build UI
def list_options(modality):
    if not HAS_ARCHIVE:
        return []
    return [p.replace(EXTRACT_DIR + "/", "") for p in INDEX.get(modality, [])][:500]

def resolve_path(rel_path):
    if rel_path is None or rel_path == "":
        return None
    return os.path.join(EXTRACT_DIR, rel_path)

def infer_one(modality, rel_file, free_text):
    if modality == "text" and (free_text and free_text.strip()):
        x, feats, preview = make_feature_vector("text", free_text)
        y = predict_from_x(x)
        log_step("single", "text", "free_text", y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats
    else:
        path = resolve_path(rel_file)
        if not path or not os.path.exists(path):
            return "Pick a file from the dropdown (or paste text).", 0,0,0,0, {}
        x, feats, preview = make_feature_vector(modality, path)
        y = predict_from_x(x)
        log_step("single", modality, rel_file, y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats

def do_export():
    path = export_log()
    return path

with gr.Blocks(title="HoloSyn UI") as demo:
    gr.Markdown("## HoloSyn Visual Interface (Local Distilled Model)")
    if not HAS_ARCHIVE:
        gr.Markdown("⚠️ `Archive.zip` not found. Upload it to `/mnt/data/Archive.zip` for file browsing. Text mode still works.")

    with gr.Tab("Single"):
        with gr.Row():
            modality = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality")
            file_dd = gr.Dropdown(choices=list_options("text"), label="File from Archive (optional)")
        free_text = gr.Textbox(lines=6, label="Text input (only used if modality=text and non-empty)")
        run_btn = gr.Button("Run inference")

        with gr.Row():
            preview = gr.Image(label="Preview (image/video frame)", type="pil")
            preview_txt = gr.Textbox(label="Preview text (if not an image)", lines=8)

        with gr.Row():
            val = gr.Slider(0,1, step=0.001, label="Valence", interactive=False)
            aro = gr.Slider(0,1, step=0.001, label="Arousal", interactive=False)
            calm = gr.Slider(0,1, step=0.001, label="Calm", interactive=False)
            trust = gr.Slider(0,1, step=0.001, label="Trust", interactive=False)
        feats_json = gr.JSON(label="Extracted features used by student")

        def refresh_files(m):
            return gr.update(choices=list_options(m), value=None)

        modality.change(refresh_files, inputs=[modality], outputs=[file_dd])

        def process_single(m, f, txt):
            prev_obj, v, a, c, t, feats = infer_one(m, f, txt)
            if isinstance(prev_obj, Image.Image):
                return prev_obj, "", v, a, c, t, feats
            else:
                return None, str(prev_obj), v, a, c, t, feats

        run_btn.click(
            fn=process_single,
            inputs=[modality, file_dd, free_text],
            outputs=[preview, preview_txt, val, aro, calm, trust, feats_json]
        )

    with gr.Tab("Two-peer synchrony"):
        gr.Markdown("Run two inputs (A/B) and compute synchrony on the model outputs.")
        with gr.Row():
            modalityA = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality A")
            fileA = gr.Dropdown(choices=list_options("text"), label="File A")
        textA = gr.Textbox(lines=4, label="Text A (used if Modality A=text and non-empty)")

        with gr.Row():
            modalityB = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality B")
            fileB = gr.Dropdown(choices=list_options("text"), label="File B")
        textB = gr.Textbox(lines=4, label="Text B (used if Modality B=text and non-empty)")

        run_pair = gr.Button("Run pair + synchrony")

        with gr.Row():
            with gr.Column():
                prevA_img = gr.Image(label="Preview A (Image/Video)", type="pil")
                prevA_txt = gr.Textbox(label="Preview A (Text/Other)", lines=4)
            with gr.Column():
                prevB_img = gr.Image(label="Preview B (Image/Video)", type="pil")
                prevB_txt = gr.Textbox(label="Preview B (Text/Other)", lines=4)

        sync = gr.Slider(0,1, step=0.001, label="Synchrony (cosine similarity)", interactive=False)

        modalityA.change(lambda m: gr.update(choices=list_options(m), value=None), inputs=[modalityA], outputs=[fileA])
        modalityB.change(lambda m: gr.update(choices=list_options(m), value=None), inputs=[modalityB], outputs=[fileB])

        def process_pair(mA, fA, txtA, mB, fB, txtB):
            prevA, vA, aA, cA, tA, featsA = infer_one(mA, fA, txtA)
            prevB, vB, aB, cB, tB, featsB = infer_one(mB, fB, txtB)

            eA = np.array([vA,aA,cA,tA], dtype=np.float32)
            eB = np.array([vB,aB,cB,tB], dtype=np.float32)
            sync_val = float(np.dot(eA, eB) / (np.linalg.norm(eA)*np.linalg.norm(eB) + 1e-8))

            outA_img = prevA if isinstance(prevA, Image.Image) else None
            outA_txt = "" if isinstance(prevA, Image.Image) else str(prevA)
            outB_img = prevB if isinstance(prevB, Image.Image) else None
            outB_txt = "" if isinstance(prevB, Image.Image) else str(prevB)

            return outA_img, outA_txt, outB_img, outB_txt, sync_val

        run_pair.click(
            fn=process_pair,
            inputs=[modalityA, fileA, textA, modalityB, fileB, textB],
            outputs=[prevA_img, prevA_txt, prevB_img, prevB_txt, sync]
        )

    with gr.Tab("Export session log"):
        gr.Markdown("Exports all inference steps recorded so far to JSON.")
        export_btn = gr.Button("Export log")
        out_path = gr.Textbox(label="Saved to")
        export_btn.click(fn=do_export, inputs=[], outputs=[out_path])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d41fdbb3712dfcd5f8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [14]:
#@title 2) Optional: extract archive and index files
EXTRACT_DIR = "/content/archive_extracted_ui"
Path(EXTRACT_DIR).mkdir(parents=True, exist_ok=True)

AUDIO_EXT = {".wav",".mp3",".m4a",".flac",".ogg",".aac"}
VIDEO_EXT = {".mp4",".mov",".mkv",".webm",".avi"}
IMAGE_EXT = {".png",".jpg",".jpeg",".webp",".bmp"}
TEXT_EXT  = {".txt",".md",".json",".csv",".tsv"}

def extract_archive_if_present():
    if os.path.exists(ARCHIVE_ZIP):
        with zipfile.ZipFile(ARCHIVE_ZIP, "r") as z:
            z.extractall(EXTRACT_DIR)
        return True
    return False

def walk_files(root):
    out=[]
    for p in Path(root).rglob("*"):
        if p.is_file():
            out.append(str(p))
    return out

def bucket(path):
    ext = Path(path).suffix.lower()
    low = path.lower()
    if ext in AUDIO_EXT: return "audio"
    if ext in VIDEO_EXT: return "video"
    if ext in IMAGE_EXT: return "image"
    if ext in TEXT_EXT:
        if ext in {".json",".csv",".tsv"} and any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
            return "haptics"
        return "text"
    if any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
        return "haptics"
    return "other"

HAS_ARCHIVE = extract_archive_if_present()
INDEX = {"audio":[], "video":[], "image":[], "text":[], "haptics":[], "other":[]}
if HAS_ARCHIVE:
    for f in walk_files(EXTRACT_DIR):
        INDEX[bucket(f)].append(f)

print("Archive present:", HAS_ARCHIVE)
if HAS_ARCHIVE:
    for k in ["audio","video","image","text","haptics","other"]:
        print(k, len(INDEX[k]))

Archive present: True
audio 49
video 52
image 407
text 49
haptics 97
other 25


In [15]:
#@title 3) Feature extraction
def load_text(path, max_chars=12000):
    return open(path, "r", encoding="utf-8", errors="ignore").read()[:max_chars]

def load_haptics_any(path):
    ext = Path(path).suffix.lower()
    if ext == ".json":
        try:
            return json.load(open(path, "r", encoding="utf-8", errors="ignore"))
        except Exception:
            return {"raw": load_text(path)}
    if ext in {".csv",".tsv"}:
        sep = "," if ext==".csv" else "\t"
        try:
            df = pd.read_csv(path, sep=sep)
            return {"columns": list(df.columns), "head": df.head(200).to_dict(orient="list")}
        except Exception:
            return {"raw": load_text(path)}
    return {"raw": load_text(path)}

def featurize_text(s):
    return {
        "txt_len": float(len(s)),
        "txt_lines": float(s.count("\n")+1),
        "txt_exclaim": float(s.count("!")),
        "txt_question": float(s.count("?")),
        "txt_caps_ratio": float(sum(c.isupper() for c in s)/max(1,len(s))),
    }

def featurize_haptics(h):
    raw = json.dumps(h)[:20000].lower()
    return {
        "hapt_len": float(len(raw)),
        "hapt_has_intensity": 1.0 if "intensity" in raw else 0.0,
        "hapt_has_freq": 1.0 if ("hz" in raw or "freq" in raw) else 0.0,
    }

def audio_features(path, sr=16000, max_seconds=15):
    y, _ = librosa.load(path, sr=sr, mono=True, duration=max_seconds)
    if y.size < 10:
        return {"aud_rms":0.0,"aud_zcr":0.0,"aud_centroid":0.0,"aud_tempo":0.0}
    rms = float(np.mean(librosa.feature.rms(y=y)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    tempo = float(librosa.beat.tempo(onset_envelope=onset_env, sr=sr)[0]) if onset_env.size else 0.0
    return {"aud_rms":rms,"aud_zcr":zcr,"aud_centroid":centroid,"aud_tempo":tempo}

def image_quick_stats(path):
    im = Image.open(path).convert("RGB")
    arr = np.asarray(im).astype(np.float32)/255.0
    mean = arr.mean(axis=(0,1))
    std = arr.std(axis=(0,1))
    return {
        "img_w": float(arr.shape[1]), "img_h": float(arr.shape[0]),
        "img_mean_r": float(mean[0]), "img_mean_g": float(mean[1]), "img_mean_b": float(mean[2]),
        "img_std_r": float(std[0]), "img_std_g": float(std[1]), "img_std_b": float(std[2]),
    }

def sample_video_frames(video_path, every_n_frames=45, max_frames=4, target_size=224):
    cap = cv2.VideoCapture(video_path)
    frames=[]
    idx=0
    while cap.isOpened() and len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret: break
        if idx % every_n_frames == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h,w = frame.shape[:2]
            if max(h,w) > target_size:
                scale = target_size / max(h,w)
                frame = cv2.resize(frame, (int(w*scale), int(h*scale)))
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

def make_feature_vector(modality, path_or_text):
    feats = {}
    preview = None

    if modality == "text":
        s = path_or_text if isinstance(path_or_text, str) and "\n" in path_or_text else load_text(path_or_text)
        feats.update(featurize_text(s))
        # Compute the 768-d embedding
        emb = embedder.encode(s)
        for i in range(768):
            feats[f"w2v_{i}"] = float(emb[i])
        preview = s[:1000]

    elif modality == "haptics":
        h = load_haptics_any(path_or_text)
        feats.update(featurize_haptics(h))
        preview = json.dumps(h)[:1000]
    elif modality == "audio":
        feats.update(audio_features(path_or_text))
        preview = f"Audio file: {Path(path_or_text).name}"
    elif modality == "image":
        feats.update(image_quick_stats(path_or_text))
        preview = Image.open(path_or_text).convert("RGB")
    elif modality == "video":
        frames = sample_video_frames(path_or_text)
        feats["vid_n_frames"] = float(len(frames))
        preview = Image.fromarray(frames[0]).convert("RGB") if frames else None
    else:
        preview = f"Unsupported modality for: {path_or_text}"

    # Build full vector
    x = np.zeros((len(NUMERIC_COLS),), dtype=np.float32)
    for i, col in enumerate(NUMERIC_COLS):
        if col in feats:
            x[i] = np.float32(feats[col])
        else:
            x[i] = 0.0
    return x, feats, preview

def predict_from_x(x):
    xn = (x - MU) / SD
    xt = torch.tensor(xn, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        y = model(xt).squeeze(0).cpu().numpy().astype(np.float32)
    return y

In [16]:
#@title 4) Session logger utilities
SESSION_LOG = []

def log_step(label, modality, source, y, feats):
    rec = {
        "ts": time.time(),
        "label": label,
        "modality": modality,
        "source": source,
        "valence": float(y[0]),
        "arousal": float(y[1]),
        "calm": float(y[2]),
        "trust": float(y[3]),
        "features": {k: float(v) if isinstance(v,(int,float,np.floating)) else str(v) for k,v in feats.items()}
    }
    SESSION_LOG.append(rec)

def export_log():
    out = "/mnt/data/holosyn_session_log.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(SESSION_LOG, f, indent=2)
    return out

In [17]:
#@title 5) Build UI
def list_options(modality):
    if not HAS_ARCHIVE:
        return []
    return [p.replace(EXTRACT_DIR + "/", "") for p in INDEX.get(modality, [])][:500]

def resolve_path(rel_path):
    if rel_path is None or rel_path == "":
        return None
    return os.path.join(EXTRACT_DIR, rel_path)

def infer_one(modality, rel_file, free_text):
    if modality == "text" and (free_text and free_text.strip()):
        x, feats, preview = make_feature_vector("text", free_text)
        y = predict_from_x(x)
        log_step("single", "text", "free_text", y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats
    else:
        path = resolve_path(rel_file)
        if not path or not os.path.exists(path):
            return "Pick a file from the dropdown (or paste text).", 0,0,0,0, {}
        x, feats, preview = make_feature_vector(modality, path)
        y = predict_from_x(x)
        log_step("single", modality, rel_file, y, feats)
        return preview, float(y[0]), float(y[1]), float(y[2]), float(y[3]), feats

def do_export():
    path = export_log()
    return path

with gr.Blocks(title="HoloSyn UI") as demo:
    gr.Markdown("## HoloSyn Visual Interface (Local Distilled Model)")
    if not HAS_ARCHIVE:
        gr.Markdown("⚠️ `Archive.zip` not found. Upload it to `/mnt/data/Archive.zip` for file browsing. Text mode still works.")

    with gr.Tab("Single"):
        with gr.Row():
            modality = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality")
            file_dd = gr.Dropdown(choices=list_options("text"), label="File from Archive (optional)")
        free_text = gr.Textbox(lines=6, label="Text input (only used if modality=text and non-empty)")
        run_btn = gr.Button("Run inference")

        with gr.Row():
            preview = gr.Image(label="Preview (image/video frame)", type="pil")
            preview_txt = gr.Textbox(label="Preview text (if not an image)", lines=8)

        with gr.Row():
            val = gr.Slider(0,1, step=0.001, label="Valence", interactive=False)
            aro = gr.Slider(0,1, step=0.001, label="Arousal", interactive=False)
            calm = gr.Slider(0,1, step=0.001, label="Calm", interactive=False)
            trust = gr.Slider(0,1, step=0.001, label="Trust", interactive=False)
        feats_json = gr.JSON(label="Extracted features used by student")

        def refresh_files(m):
            return gr.update(choices=list_options(m), value=None)

        modality.change(refresh_files, inputs=[modality], outputs=[file_dd])

        def process_single(m, f, txt):
            prev_obj, v, a, c, t, feats = infer_one(m, f, txt)
            if isinstance(prev_obj, Image.Image):
                return prev_obj, "", v, a, c, t, feats
            else:
                return None, str(prev_obj), v, a, c, t, feats

        run_btn.click(
            fn=process_single,
            inputs=[modality, file_dd, free_text],
            outputs=[preview, preview_txt, val, aro, calm, trust, feats_json]
        )

    with gr.Tab("Two-peer synchrony"):
        gr.Markdown("Run two inputs (A/B) and compute synchrony on the model outputs.")
        with gr.Row():
            modalityA = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality A")
            fileA = gr.Dropdown(choices=list_options("text"), label="File A")
        textA = gr.Textbox(lines=4, label="Text A (used if Modality A=text and non-empty)")

        with gr.Row():
            modalityB = gr.Dropdown(choices=["text","audio","image","video","haptics"], value="text", label="Modality B")
            fileB = gr.Dropdown(choices=list_options("text"), label="File B")
        textB = gr.Textbox(lines=4, label="Text B (used if Modality B=text and non-empty)")

        run_pair = gr.Button("Run pair + synchrony")

        with gr.Row():
            with gr.Column():
                prevA_img = gr.Image(label="Preview A (Image/Video)", type="pil")
                prevA_txt = gr.Textbox(label="Preview A (Text/Other)", lines=4)
            with gr.Column():
                prevB_img = gr.Image(label="Preview B (Image/Video)", type="pil")
                prevB_txt = gr.Textbox(label="Preview B (Text/Other)", lines=4)

        sync = gr.Slider(0,1, step=0.001, label="Synchrony (cosine similarity)", interactive=False)

        modalityA.change(lambda m: gr.update(choices=list_options(m), value=None), inputs=[modalityA], outputs=[fileA])
        modalityB.change(lambda m: gr.update(choices=list_options(m), value=None), inputs=[modalityB], outputs=[fileB])

        def process_pair(mA, fA, txtA, mB, fB, txtB):
            prevA, vA, aA, cA, tA, featsA = infer_one(mA, fA, txtA)
            prevB, vB, aB, cB, tB, featsB = infer_one(mB, fB, txtB)

            eA = np.array([vA,aA,cA,tA], dtype=np.float32)
            eB = np.array([vB,aB,cB,tB], dtype=np.float32)
            sync_val = float(np.dot(eA, eB) / (np.linalg.norm(eA)*np.linalg.norm(eB) + 1e-8))

            outA_img = prevA if isinstance(prevA, Image.Image) else None
            outA_txt = "" if isinstance(prevA, Image.Image) else str(prevA)
            outB_img = prevB if isinstance(prevB, Image.Image) else None
            outB_txt = "" if isinstance(prevB, Image.Image) else str(prevB)

            return outA_img, outA_txt, outB_img, outB_txt, sync_val

        run_pair.click(
            fn=process_pair,
            inputs=[modalityA, fileA, textA, modalityB, fileB, textB],
            outputs=[prevA_img, prevA_txt, prevB_img, prevB_txt, sync]
        )

    with gr.Tab("Export session log"):
        gr.Markdown("Exports all inference steps recorded so far to JSON.")
        export_btn = gr.Button("Export log")
        out_path = gr.Textbox(label="Saved to")
        export_btn.click(fn=do_export, inputs=[], outputs=[out_path])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e2b4bc8cb7a2cd3eef.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [18]:
import os
import json
import traceback
import numpy as np
import torch
import gradio as gr
import matplotlib.pyplot as plt

# ==========================================
# 1. CONFIGURATION & FILE DISCOVERY
# ==========================================
# Add your local or colab paths here. The app will search these paths.
SEARCH_PATHS = ["/mnt/data", "./", "/content"]
MODEL_CANDIDATES = [
    "student_distilled_heads_hf.torchscript (1).pt",
    "student_distilled_heads_hf.torchscript.pt",
    "student_distilled_export.torchscript.pt",
    "student_distilled_heads.torchscript.pt",
    "holosyn_heads.torchscript.pt"
]
NORM_FILE = "student_norm_hf.json"

def find_file(filename):
    for path in SEARCH_PATHS:
        full_path = os.path.join(path, filename)
        if os.path.exists(full_path):
            return full_path
    return None

# ==========================================
# 2. CORE LOADERS
# ==========================================
def load_system():
    debug_logs = []
    model = None
    norms = None
    model_path = None

    # Load Norms
    norm_path = find_file(NORM_FILE)
    if norm_path:
        try:
            with open(norm_path, 'r') as f:
                norms = json.load(f)
            debug_logs.append(f"[OK] Loaded norms from {norm_path}")
        except Exception as e:
            debug_logs.append(f"[ERROR] Failed to load norms: {str(e)}")
    else:
        debug_logs.append(f"[WARN] Normalization file '{NORM_FILE}' not found.")

    # Load Model
    for candidate in MODEL_CANDIDATES:
        cand_path = find_file(candidate)
        if cand_path:
            try:
                model = torch.jit.load(cand_path)
                model.eval()
                model_path = cand_path
                debug_logs.append(f"[OK] Successfully loaded TorchScript model: {cand_path}")
                break
            except Exception as e:
                debug_logs.append(f"[ERROR] Found {cand_path} but failed to load: {str(e)}")

    if not model:
        debug_logs.append("[FATAL] No valid model could be loaded from candidate list.")

    return model, norms, model_path, "\n".join(debug_logs)

# Initialize globally
GLOBAL_MODEL, GLOBAL_NORMS, GLOBAL_MODEL_PATH, INIT_LOGS = load_system()

# ==========================================
# 3. FEATURE EXTRACTION & NORMALIZATION
# ==========================================
def extract_features(text, audio, image):
    """
    Simulates extracting the 53 features mapped in student_norm_hf.json
    In production, replace these mock values with actual processing (Librosa, CV2, Word2Vec, etc.)
    """
    # Base feature dictionary based on the JSON snippet provided
    features = {
        "txt_len": len(text) if text else 0,
        "txt_lines": text.count('\n') + 1 if text else 0,
        "txt_exclaim": text.count('!') if text else 0,
        "txt_question": text.count('?') if text else 0,
        "txt_caps_ratio": sum(1 for c in text if c.isupper()) / (len(text) + 1e-5) if text else 0,

        "hapt_len": 0, "hapt_has_intensity": 0, "hapt_has_freq": 0,

        "aud_rms": 0.5 if audio else 0,
        "aud_zcr": 0.1 if audio else 0,
        "aud_centroid": 1500 if audio else 0,
        "aud_tempo": 120 if audio else 0,

        "img_w": 512 if image is not None else 0,
        "img_h": 512 if image is not None else 0,
        "img_mean_r": 128 if image is not None else 0,
        "img_mean_g": 128 if image is not None else 0,
        "img_mean_b": 128 if image is not None else 0,
        "img_std_r": 30 if image is not None else 0,
        "img_std_g": 30 if image is not None else 0,
        "img_std_b": 30 if image is not None else 0,

        "vid_n_frames": 0
    }

    # Mock Word2Vec/Embeddings (w2v_0 to w2v_31)
    for i in range(32):
        features[f"w2v_{i}"] = np.random.normal(0, 1) if text else 0.0

    return features

def apply_normalization(features_dict, norms):
    """Applies min-max or z-score normalization based on the JSON dict."""
    if not norms or "numeric_cols" not in norms:
        # Pass-through if no norms available
        return list(features_dict.values())

    vector = []
    for col in norms["numeric_cols"]:
        val = features_dict.get(col, 0.0)
        # Apply normalization if std/mean exist in your JSON structure
        # Assuming format like: norms["means"][col], norms["stds"][col]
        # For safety in this script, we pass raw if exact metrics aren't parsed
        mean = norms.get("means", {}).get(col, 0.0)
        std = norms.get("stds", {}).get(col, 1.0)

        # Avoid division by zero
        std = std if std > 1e-6 else 1.0
        norm_val = (val - mean) / std
        vector.append(norm_val)

    return vector

# ==========================================
# 4. INFERENCE ENGINE & DEBUGGER
# ==========================================
def process_inference(text, audio, image):
    debug_info = {}

    try:
        # 1. Extraction
        raw_features = extract_features(text, audio, image)
        debug_info["raw_features"] = raw_features

        # 2. Normalization
        feature_vector = apply_normalization(raw_features, GLOBAL_NORMS)
        debug_info["normalized_vector"] = feature_vector
        debug_info["vector_length"] = len(feature_vector)

        if GLOBAL_MODEL is None:
            raise ValueError("No model loaded. Check System Logs.")

        # 3. Tensor Prep
        input_tensor = torch.tensor([feature_vector], dtype=torch.float32)
        debug_info["input_tensor_shape"] = list(input_tensor.shape)

        # 4. Inference
        with torch.no_grad():
            output = GLOBAL_MODEL(input_tensor)

        # 5. Parse Output
        # Assuming output is a tensor of shape (1, 4) mapping to [Valence, Arousal, Calm, Trust]
        # Adjust parsing based on your specific model head architecture.
        if isinstance(output, tuple):
            output = output[0] # Handle tuple returns

        out_array = output.squeeze().cpu().numpy().tolist()
        debug_info["raw_output_tensor"] = out_array

        # Safe extraction of 4 expected outputs
        val = out_array[0] if len(out_array) > 0 else 0.0
        aro = out_array[1] if len(out_array) > 1 else 0.0
        cal = out_array[2] if len(out_array) > 2 else 0.0
        tru = out_array[3] if len(out_array) > 3 else 0.0

        results = {
            "Valence": round(float(val), 3),
            "Arousal": round(float(aro), 3),
            "Calm": round(float(cal), 3),
            "Trust": round(float(tru), 3)
        }

        return results, json.dumps(debug_info, indent=2)

    except Exception as e:
        debug_info["ERROR"] = str(e)
        debug_info["TRACEBACK"] = traceback.format_exc()
        return {"Error": "Failed"}, json.dumps(debug_info, indent=2)

# ==========================================
# 5. GRADIO USER INTERFACE
# ==========================================
def create_ui():
    with gr.Blocks(theme=gr.themes.Soft(), title="HoloSyn Interface") as app:
        gr.Markdown("# 🧠 HoloSyn Multimodal UI")
        gr.Markdown("Integrates your Distilled TorchScript Model for Valence, Arousal, Calm, and Trust.")

        with gr.Tabs():
            # TAB 1: Main Inference
            with gr.Tab("Single Target Inference"):
                with gr.Row():
                    with gr.Column(scale=1):
                        gr.Markdown("### Input Modalities")
                        txt_in = gr.Textbox(label="Text Input (Transcript/Chat)", lines=3)
                        aud_in = gr.Audio(label="Audio Input (Voice)", type="filepath")
                        img_in = gr.Image(label="Visual Input (Face/Context)", type="filepath")
                        btn_run = gr.Button("Analyze Modalities", variant="primary")

                    with gr.Column(scale=1):
                        gr.Markdown("### Emotional / Cognitive Output")
                        out_label = gr.Label(num_top_classes=4, label="Model Predictions")

            # TAB 2: Synchrony (A/B Test)
            with gr.Tab("Two-Peer Synchrony"):
                gr.Markdown("Compare two individuals or time-steps simultaneously.")
                with gr.Row():
                    with gr.Column():
                        gr.Markdown("### Peer A")
                        txt_a = gr.Textbox(label="Text A", lines=2)
                        btn_a = gr.Button("Analyze Peer A")
                        out_a = gr.Label(label="State A")
                    with gr.Column():
                        gr.Markdown("### Peer B")
                        txt_b = gr.Textbox(label="Text B", lines=2)
                        btn_b = gr.Button("Analyze Peer B")
                        out_b = gr.Label(label="State B")

            # TAB 3: Debug & Diagnostics Interface
            with gr.Tab("System Debug & Diagnostics"):
                gr.Markdown("Use this interface if dimensions mismatch or tensors fail to load.")
                with gr.Row():
                    with gr.Column():
                        gr.Markdown("### Initialization Logs")
                        gr.Textbox(value=INIT_LOGS, lines=6, interactive=False, label="Boot Sequence")

                        gr.Markdown("### Model Info")
                        gr.JSON(value={
                            "Model Path": GLOBAL_MODEL_PATH,
                            "Has Norms": GLOBAL_NORMS is not None,
                            "Expected Input Dimension": len(GLOBAL_NORMS["numeric_cols"]) if GLOBAL_NORMS else "Unknown"
                        }, label="Environment State")

                    with gr.Column():
                        gr.Markdown("### Last Inference Debug Trace")
                        debug_out = gr.Code(language="json", label="Tensor Details, Shapes, and Errors")

        # Event Wirings
        btn_run.click(fn=process_inference, inputs=[txt_in, aud_in, img_in], outputs=[out_label, debug_out])

        # Synchrony wirings (Simplified to just text for quick A/B testing)
        btn_a.click(fn=lambda t: process_inference(t, None, None)[0], inputs=[txt_a], outputs=[out_a])
        btn_b.click(fn=lambda t: process_inference(t, None, None)[0], inputs=[txt_b], outputs=[out_b])

    return app

if __name__ == "__main__":
    # If running inside a notebook cell, this will mount the app below the cell.
    app = create_ui()
    app.launch(debug=True, share=True)

/tmp/ipython-input-417901468.py:195: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="HoloSyn Interface") as app:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8f90bac3eb6adb20e5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://50c6bff44aa90dc61f.gradio.live
Killing tunnel 127.0.0.1:7861 <> https://54628bd43af5f1a363.gradio.live
Killing tunnel 127.0.0.1:7862 <> https://d41fdbb3712dfcd5f8.gradio.live
Killing tunnel 127.0.0.1:7863 <> https://e2b4bc8cb7a2cd3eef.gradio.live
Killing tunnel 127.0.0.1:7864 <> https://8f90bac3eb6adb20e5.gradio.live


In [19]:
# ==============================================================================
# HoloSyn Visual Interface (Updated & Debug-Ready)
# Copy and paste this entire code block into your Jupyter/Colab notebook cell.
# ==============================================================================

import os
import json
import torch
import numpy as np
import gradio as gr
import traceback

# ------------------------------------------------------------------------------
# 1. Configuration & Path Resolution
# ------------------------------------------------------------------------------
# We search common environments (Local, Colab, Custom Mounts)
POTENTIAL_MODEL_PATHS = [
    "student_distilled_heads_hf.torchscript.pt",
    "student_distilled_heads_hf.torchscript (1).pt",
    "student_distilled_export.torchscript.pt",
    "/content/student_distilled_heads_hf.torchscript.pt",
    "/mnt/data/student_distilled_heads_hf.torchscript.pt"
]

POTENTIAL_NORM_PATHS = [
    "student_norm_hf.json",
    "/content/student_norm_hf.json",
    "/mnt/data/student_norm_hf.json"
]

def find_file(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    return None

MODEL_PATH = find_file(POTENTIAL_MODEL_PATHS)
NORM_PATH = find_file(POTENTIAL_NORM_PATHS)

# ------------------------------------------------------------------------------
# 2. Global State & Initialization
# ------------------------------------------------------------------------------
model = None
norm_data = None
feature_columns = []
means = []
stds = []

def initialize_system():
    global model, norm_data, feature_columns, means, stds
    logs = ["=== System Initialization ==="]

    # Load Normalization Data
    if NORM_PATH:
        logs.append(f"[OK] Found norm file at: {NORM_PATH}")
        try:
            with open(NORM_PATH, 'r') as f:
                norm_data = json.load(f)
            # Support different JSON structures
            feature_columns = norm_data.get('numeric_cols', [])

            # If means/stds aren't explicitly saved as arrays, fallback to dictionary mapping or zeros
            if 'means' in norm_data and 'stds' in norm_data:
                means = np.array(norm_data['means'])
                stds = np.array(norm_data['stds'])
            else:
                logs.append("[WARNING] 'means' or 'stds' arrays not found in JSON. Defaulting to 0 mean / 1 std.")
                means = np.zeros(len(feature_columns))
                stds = np.ones(len(feature_columns))

            logs.append(f"[OK] Extracted {len(feature_columns)} feature columns from norm file.")
        except Exception as e:
            logs.append(f"[ERROR] Failed to load norm JSON: {e}")
    else:
        logs.append("[ERROR] Normalization JSON not found. Please upload student_norm_hf.json.")

    # Load TorchScript Model
    if MODEL_PATH:
        logs.append(f"[OK] Found model at: {MODEL_PATH}")
        try:
            model = torch.jit.load(MODEL_PATH, map_location=torch.device('cpu'))
            model.eval()
            logs.append("[OK] TorchScript model loaded successfully.")
        except Exception as e:
            logs.append(f"[ERROR] Failed to load model: {e}")
    else:
        logs.append("[ERROR] TorchScript model not found. Please upload the .pt file.")

    return "\n".join(logs)

# Run init immediately
init_logs = initialize_system()

# ------------------------------------------------------------------------------
# 3. Processing & Inference Engine
# ------------------------------------------------------------------------------
def process_inputs(text, audio_path, image_path):
    logs = ["=== Inference Execution ==="]

    if model is None:
        logs.append("[FATAL ERROR] Model is not loaded. Cannot predict.")
        return 0, 0, 0, 0, "\n".join(logs)

    if len(feature_columns) == 0:
        logs.append("[FATAL ERROR] Feature dimension unknown. Norm data missing.")
        return 0, 0, 0, 0, "\n".join(logs)

    try:
        # 1. Initialize empty feature vector matching the EXACT size required by the distilled head
        raw_features = np.zeros(len(feature_columns), dtype=np.float32)
        logs.append(f"Initialized raw feature vector of shape: {raw_features.shape}")

        # 2. Extract mock/basic features based on inputs provided
        # In a real scenario, you map extracted features to their exact index in `feature_columns`
        if text and len(text.strip()) > 0:
            logs.append("Processing Text Input...")
            if "txt_len" in feature_columns:
                idx = feature_columns.index("txt_len")
                raw_features[idx] = len(text)
            if "txt_exclaim" in feature_columns:
                idx = feature_columns.index("txt_exclaim")
                raw_features[idx] = text.count("!")

        if audio_path:
            logs.append(f"Processing Audio File: {os.path.basename(audio_path)}...")
            # Placeholder for actual Librosa extraction (RMS, ZCR, etc.)
            if "aud_rms" in feature_columns:
                idx = feature_columns.index("aud_rms")
                raw_features[idx] = 0.5 # Mock value

        if image_path:
            logs.append("Processing Image File...")
            if "img_mean_r" in feature_columns:
                idx = feature_columns.index("img_mean_r")
                raw_features[idx] = 128.0 # Mock value

        # 3. Normalize the vector
        logs.append("Applying normalization (subtract mean, divide by std)...")
        normalized_features = (raw_features - means) / (stds + 1e-8)

        # 4. Convert to Tensor
        input_tensor = torch.tensor(normalized_features, dtype=torch.float32).unsqueeze(0)
        logs.append(f"Input tensor prepared. Shape: {input_tensor.shape}")

        # 5. Forward Pass
        logs.append("Running model inference...")
        with torch.no_grad():
            output = model(input_tensor)

        logs.append(f"Raw Output Tensor: {output}")

        # 6. Parse Outputs (Assuming standard 4-dimension output: Valence, Arousal, Calm, Trust)
        # Adjust indices if your distilled head outputs a different order
        out_array = output.squeeze().numpy()

        if out_array.size >= 4:
            valence = float(out_array[0])
            arousal = float(out_array[1])
            calm    = float(out_array[2])
            trust   = float(out_array[3])
        else:
            # Fallback if the output size is unexpectedly small
            valence = float(out_array[0]) if out_array.size > 0 else 0.0
            arousal = float(out_array[1]) if out_array.size > 1 else 0.0
            calm    = float(out_array[2]) if out_array.size > 2 else 0.0
            trust   = float(out_array[3]) if out_array.size > 3 else 0.0
            logs.append("[WARNING] Model output size was smaller than 4 dimensions.")

        logs.append("Inference complete.")

        # Optional: scale values to 0-1 or 0-100 for gauges if they are raw logits
        # valence = torch.sigmoid(torch.tensor(valence)).item() * 100

        return valence, arousal, calm, trust, "\n".join(logs)

    except Exception as e:
        error_trace = traceback.format_exc()
        logs.append(f"[EXCEPTION CAUGHT]\n{error_trace}")
        return 0, 0, 0, 0, "\n".join(logs)

# ------------------------------------------------------------------------------
# 4. Gradio User Interface
# ------------------------------------------------------------------------------
css = """
.gradio-container { max-width: 900px !important; }
.output-gauge { font-size: 2em; font-weight: bold; text-align: center; }
"""

with gr.Blocks(theme=gr.themes.Soft(), css=css) as demo:
    gr.Markdown("# 🧠 HoloSyn Distilled Model Interface")
    gr.Markdown("Test your distilled TorchScript heads with multi-modal inputs. The system dynamically scales to your `student_norm_hf.json` configuration.")

    with gr.Tabs():

        # --- TAB 1: Main Inference ---
        with gr.Tab("Single Peer Analysis"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### Input Modalities")
                    txt_input = gr.Textbox(lines=3, label="Text / Transcript", placeholder="Type something...")
                    aud_input = gr.Audio(type="filepath", label="Audio Upload")
                    img_input = gr.Image(type="filepath", label="Image Upload")

                    analyze_btn = gr.Button("Analyze Signal", variant="primary")

                with gr.Column(scale=1):
                    gr.Markdown("### Synthesized Outputs")
                    with gr.Row():
                        val_out = gr.Number(label="Valence", elem_classes="output-gauge")
                        aro_out = gr.Number(label="Arousal", elem_classes="output-gauge")
                    with gr.Row():
                        clm_out = gr.Number(label="Calm", elem_classes="output-gauge")
                        trs_out = gr.Number(label="Trust", elem_classes="output-gauge")

            with gr.Accordion("Debug Log & Execution Trace", open=False):
                debug_out = gr.Textbox(lines=10, label="Console Output", value=init_logs, interactive=False)

        # --- TAB 2: System Status ---
        with gr.Tab("System & Files"):
            gr.Markdown("### Current Environment Status")
            gr.Textbox(label="Resolved Model Path", value=str(MODEL_PATH), interactive=False)
            gr.Textbox(label="Resolved Norm Path", value=str(NORM_PATH), interactive=False)
            gr.Textbox(label="Feature Space Dimension", value=str(len(feature_columns)), interactive=False)

            reinit_btn = gr.Button("Reload Model & Configs")

            def manual_reinit():
                logs = initialize_system()
                return logs, str(MODEL_PATH), str(NORM_PATH), str(len(feature_columns))

            status_out = gr.Textbox(label="Reload Log", lines=5)
            reinit_btn.click(manual_reinit, outputs=[status_out,
                                                     gr.Textbox(label="Resolved Model Path"),
                                                     gr.Textbox(label="Resolved Norm Path"),
                                                     gr.Textbox(label="Feature Space Dimension")])

    # Connect execution logic
    analyze_btn.click(
        fn=process_inputs,
        inputs=[txt_input, aud_input, img_input],
        outputs=[val_out, aro_out, clm_out, trs_out, debug_out]
    )

# ------------------------------------------------------------------------------
# 5. Launch
# ------------------------------------------------------------------------------
if __name__ == "__main__":
    # In Colab, inline launch is preferred.
    # share=True provides a public URL for easy testing.
    demo.launch(debug=True, share=True, inline=True)

/tmp/ipython-input-1962647530.py:189: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=css) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://51a9fb2a58e243f05c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7864 <> https://51a9fb2a58e243f05c.gradio.live


In [20]:
# ==============================================================================
# HoloSyn Visual Interface (Updated & Debug-Ready)
# Copy and paste this entire code block into your Jupyter/Colab notebook cell.
# ==============================================================================

import os
import json
import torch
import numpy as np
import gradio as gr
import traceback

# --- New Integrations (Transformers, Brian2, Cirq) ---
import matplotlib.pyplot as plt

try:
    from transformers import pipeline
    HAS_TRANSFORMERS = True
except ImportError:
    HAS_TRANSFORMERS = False

try:
    import brian2 as b2
    HAS_BRIAN2 = True
except ImportError:
    HAS_BRIAN2 = False

try:
    import cirq
    HAS_CIRQ = True
    try:
        import qsimcirq
        HAS_QSIM = True
    except ImportError:
        HAS_QSIM = False
except ImportError:
    HAS_CIRQ = False
    HAS_QSIM = False

# Global AI Pipeline (Lightweight open-source extraction)
ai_pipeline = None
if HAS_TRANSFORMERS:
    try:
        ai_pipeline = pipeline("feature-extraction", model="prajjwal1/bert-tiny")
    except Exception as e:
        print(f"Failed to load AI pipeline: {e}")

# ------------------------------------------------------------------------------
# 1. Configuration & Path Resolution
# ------------------------------------------------------------------------------
# We search common environments (Local, Colab, Custom Mounts)
POTENTIAL_MODEL_PATHS = [
    "student_distilled_heads_hf.torchscript.pt",
    "student_distilled_heads_hf.torchscript (1).pt",
    "student_distilled_export.torchscript.pt",
    "/content/student_distilled_heads_hf.torchscript.pt",
    "/mnt/data/student_distilled_heads_hf.torchscript.pt"
]

POTENTIAL_NORM_PATHS = [
    "student_norm_hf.json",
    "/content/student_norm_hf.json",
    "/mnt/data/student_norm_hf.json"
]

def find_file(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    return None

MODEL_PATH = find_file(POTENTIAL_MODEL_PATHS)
NORM_PATH = find_file(POTENTIAL_NORM_PATHS)

# ------------------------------------------------------------------------------
# 2. Global State & Initialization
# ------------------------------------------------------------------------------
model = None
norm_data = None
feature_columns = []
means = []
stds = []

def initialize_system():
    global model, norm_data, feature_columns, means, stds
    logs = ["=== System Initialization ==="]

    # Load Normalization Data
    if NORM_PATH:
        logs.append(f"[OK] Found norm file at: {NORM_PATH}")
        try:
            with open(NORM_PATH, 'r') as f:
                norm_data = json.load(f)
            # Support different JSON structures
            feature_columns = norm_data.get('numeric_cols', [])

            # If means/stds aren't explicitly saved as arrays, fallback to dictionary mapping or zeros
            if 'means' in norm_data and 'stds' in norm_data:
                means = np.array(norm_data['means'])
                stds = np.array(norm_data['stds'])
            else:
                logs.append("[WARNING] 'means' or 'stds' arrays not found in JSON. Defaulting to 0 mean / 1 std.")
                means = np.zeros(len(feature_columns))
                stds = np.ones(len(feature_columns))

            logs.append(f"[OK] Extracted {len(feature_columns)} feature columns from norm file.")
        except Exception as e:
            logs.append(f"[ERROR] Failed to load norm JSON: {e}")
    else:
        logs.append("[ERROR] Normalization JSON not found. Please upload student_norm_hf.json.")

    # Load TorchScript Model
    if MODEL_PATH:
        logs.append(f"[OK] Found model at: {MODEL_PATH}")
        try:
            model = torch.jit.load(MODEL_PATH, map_location=torch.device('cpu'))
            model.eval()
            logs.append("[OK] TorchScript model loaded successfully.")
        except Exception as e:
            logs.append(f"[ERROR] Failed to load model: {e}")
    else:
        logs.append("[ERROR] TorchScript model not found. Please upload the .pt file.")

    return "\n".join(logs)

# Run init immediately
init_logs = initialize_system()

# ------------------------------------------------------------------------------
# 3. Processing & Inference Engine
# ------------------------------------------------------------------------------
def process_inputs(text, audio_path, image_path):
    logs = ["=== Inference Execution ==="]

    if model is None:
        logs.append("[FATAL ERROR] Model is not loaded. Cannot predict.")
        return 0, 0, 0, 0, "\n".join(logs)

    if len(feature_columns) == 0:
        logs.append("[FATAL ERROR] Feature dimension unknown. Norm data missing.")
        return 0, 0, 0, 0, "\n".join(logs)

    try:
        # 1. Initialize empty feature vector matching the EXACT size required by the distilled head
        raw_features = np.zeros(len(feature_columns), dtype=np.float32)
        logs.append(f"Initialized raw feature vector of shape: {raw_features.shape}")

        # --- Advanced AI -> SNN -> Quantum Distillation Pipeline ---
        fig_snn, fig_quantum = None, None

        # Step 1: Open Source AI (Transformers)
        ai_embeddings = None
        if HAS_TRANSFORMERS and ai_pipeline and text:
            logs.append("Extracting Open Source AI Features (Transformers)...")
            try:
                out = ai_pipeline(text)
                ai_embeddings = np.mean(out[0], axis=0) # Pool sequence
                logs.append(f"AI Embedding extracted, shape: {ai_embeddings.shape}")
            except Exception as e:
                logs.append(f"[AI Error]: {e}")

        # Step 2: Brian2 (SNN)
        snn_spikes = None
        if HAS_BRIAN2 and ai_embeddings is not None:
            logs.append("Running Brian2 Spiking Neural Network Distillation...")
            try:
                b2.start_scope()
                # Take first 10 dims, map to Hz
                rates = np.abs(ai_embeddings[:10]) * 50 * b2.Hz
                num_inputs = len(rates)

                P = b2.PoissonGroup(num_inputs, rates=rates)
                eqs = '''dv/dt = (-v)/(10*ms) : 1 (unless refractory)'''
                G = b2.NeuronGroup(num_inputs, eqs, threshold='v>1', reset='v=0', refractory=2*b2.ms, method='exact')
                S = b2.Synapses(P, G, on_pre='v += 0.5')
                S.connect(j='i')
                M = b2.SpikeMonitor(G)

                b2.run(50*b2.ms)
                snn_spikes = M.count[:]
                logs.append(f"SNN Spike Counts generated: {snn_spikes}")

                if len(M.t) > 0:
                    fig_snn = plt.figure(figsize=(5, 3))
                    plt.plot(M.t/b2.ms, M.i, '.k')
                    plt.title('Brian2 SNN Raster Plot')
                    plt.xlabel('Time (ms)')
                    plt.ylabel('Neuron Index')
                    plt.tight_layout()
            except Exception as e:
                logs.append(f"[Brian2 Error]: {e}")

        # Step 3: Cirq & qsimcirq
        quantum_results = None
        if HAS_CIRQ and snn_spikes is not None:
            logs.append("Encoding SNN Spikes to Quantum Circuit (Cirq/qsimcirq)...")
            try:
                qubits = cirq.LineQubit.range(4)
                circuit = cirq.Circuit()

                # Encode spikes as RX rotations
                for i, q in enumerate(qubits):
                    angle = float(snn_spikes[i % len(snn_spikes)]) * np.pi / 10.0
                    circuit.append(cirq.rx(angle)(q))

                # Entangle
                circuit.append(cirq.CNOT(qubits[0], qubits[1]))
                circuit.append(cirq.CNOT(qubits[2], qubits[3]))
                circuit.append(cirq.measure(*qubits, key='result'))

                # Simulate
                if HAS_QSIM:
                    logs.append("Simulating using Google qsimcirq...")
                    simulator = qsimcirq.QSimSimulator()
                else:
                    logs.append("Simulating using standard Cirq...")
                    simulator = cirq.Simulator()

                q_res = simulator.run(circuit, repetitions=100)
                quantum_results = q_res.histogram(key='result')
                logs.append(f"Quantum Measurement Histogram: {quantum_results}")

                fig_quantum = plt.figure(figsize=(5, 3))
                cirq.plot_state_histogram(q_res, plt.gca())
                plt.title('Quantum State Histogram')
                plt.tight_layout()
            except Exception as e:
                logs.append(f"[Quantum Error]: {e}")

        # 2. Extract mock/basic features based on inputs provided
        # In a real scenario, you map extracted features to their exact index in `feature_columns`
        if text and len(text.strip()) > 0:
            logs.append("Processing Text Input (Distilling from AI/SNN/Quantum)...")
            if "txt_len" in feature_columns:
                idx = feature_columns.index("txt_len")
                # Distill advanced features down to the expected representation space
                if quantum_results:
                    raw_features[idx] = sum(quantum_results.values()) / max(1, len(quantum_results))
                elif snn_spikes is not None:
                    raw_features[idx] = float(np.mean(snn_spikes))
                else:
                    raw_features[idx] = len(text)
            if "txt_exclaim" in feature_columns:
                idx = feature_columns.index("txt_exclaim")
                raw_features[idx] = text.count("!")

        if audio_path:
            logs.append(f"Processing Audio File: {os.path.basename(audio_path)}...")
            # Placeholder for actual Librosa extraction (RMS, ZCR, etc.)
            if "aud_rms" in feature_columns:
                idx = feature_columns.index("aud_rms")
                raw_features[idx] = 0.5 # Mock value

        if image_path:
            logs.append("Processing Image File...")
            if "img_mean_r" in feature_columns:
                idx = feature_columns.index("img_mean_r")
                raw_features[idx] = 128.0 # Mock value

        # 3. Normalize the vector
        logs.append("Applying normalization (subtract mean, divide by std)...")
        normalized_features = (raw_features - means) / (stds + 1e-8)

        # 4. Convert to Tensor
        input_tensor = torch.tensor(normalized_features, dtype=torch.float32).unsqueeze(0)
        logs.append(f"Input tensor prepared. Shape: {input_tensor.shape}")

        # 5. Forward Pass
        logs.append("Running model inference...")
        with torch.no_grad():
            output = model(input_tensor)

        logs.append(f"Raw Output Tensor: {output}")

        # 6. Parse Outputs (Assuming standard 4-dimension output: Valence, Arousal, Calm, Trust)
        # Adjust indices if your distilled head outputs a different order
        out_array = output.squeeze().numpy()

        if out_array.size >= 4:
            valence = float(out_array[0])
            arousal = float(out_array[1])
            calm    = float(out_array[2])
            trust   = float(out_array[3])
        else:
            # Fallback if the output size is unexpectedly small
            valence = float(out_array[0]) if out_array.size > 0 else 0.0
            arousal = float(out_array[1]) if out_array.size > 1 else 0.0
            calm    = float(out_array[2]) if out_array.size > 2 else 0.0
            trust   = float(out_array[3]) if out_array.size > 3 else 0.0
            logs.append("[WARNING] Model output size was smaller than 4 dimensions.")

        logs.append("Inference complete.")

        # Optional: scale values to 0-1 or 0-100 for gauges if they are raw logits
        # valence = torch.sigmoid(torch.tensor(valence)).item() * 100

        return valence, arousal, calm, trust, "\n".join(logs), fig_snn, fig_quantum

    except Exception as e:
        error_trace = traceback.format_exc()
        logs.append(f"[EXCEPTION CAUGHT]\n{error_trace}")
        return 0, 0, 0, 0, "\n".join(logs), None, None

# ------------------------------------------------------------------------------
# 4. Gradio User Interface
# ------------------------------------------------------------------------------
css = """
.gradio-container { max-width: 900px !important; }
.output-gauge { font-size: 2em; font-weight: bold; text-align: center; }
"""

with gr.Blocks(theme=gr.themes.Soft(), css=css) as demo:
    gr.Markdown("# 🧠 HoloSyn Distilled Model Interface")
    gr.Markdown("Test your distilled TorchScript heads with multi-modal inputs. The system dynamically scales to your `student_norm_hf.json` configuration.")

    with gr.Tabs():

        # --- TAB 1: Main Inference ---
        with gr.Tab("Single Peer Analysis"):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### Input Modalities")
                    txt_input = gr.Textbox(lines=3, label="Text / Transcript", placeholder="Type something...")
                    aud_input = gr.Audio(type="filepath", label="Audio Upload")
                    img_input = gr.Image(type="filepath", label="Image Upload")

                    analyze_btn = gr.Button("Analyze Signal", variant="primary")

                with gr.Column(scale=1):
                    gr.Markdown("### Synthesized Outputs")
                    with gr.Row():
                        val_out = gr.Number(label="Valence", elem_classes="output-gauge")
                        aro_out = gr.Number(label="Arousal", elem_classes="output-gauge")
                    with gr.Row():
                        clm_out = gr.Number(label="Calm", elem_classes="output-gauge")
                        trs_out = gr.Number(label="Trust", elem_classes="output-gauge")

            with gr.Accordion("Pipeline Visualizations (SNN & Quantum)", open=True):
                with gr.Row():
                    snn_plot = gr.Plot(label="Brian2 SNN Raster Plot")
                    q_plot = gr.Plot(label="Cirq / qsimcirq State Histogram")

            with gr.Accordion("Debug Log & Execution Trace", open=False):
                debug_out = gr.Textbox(lines=10, label="Console Output", value=init_logs, interactive=False)

        # --- TAB 2: System Status ---
        with gr.Tab("System & Files"):
            gr.Markdown("### Current Environment Status")
            gr.Textbox(label="Resolved Model Path", value=str(MODEL_PATH), interactive=False)
            gr.Textbox(label="Resolved Norm Path", value=str(NORM_PATH), interactive=False)
            gr.Textbox(label="Feature Space Dimension", value=str(len(feature_columns)), interactive=False)

            reinit_btn = gr.Button("Reload Model & Configs")

            def manual_reinit():
                logs = initialize_system()
                return logs, str(MODEL_PATH), str(NORM_PATH), str(len(feature_columns))

            status_out = gr.Textbox(label="Reload Log", lines=5)
            reinit_btn.click(manual_reinit, outputs=[status_out,
                                                     gr.Textbox(label="Resolved Model Path"),
                                                     gr.Textbox(label="Resolved Norm Path"),
                                                     gr.Textbox(label="Feature Space Dimension")])

    # Connect execution logic
    analyze_btn.click(
        fn=process_inputs,
        inputs=[txt_input, aud_input, img_input],
        outputs=[val_out, aro_out, clm_out, trs_out, debug_out, snn_plot, q_plot]
    )

# ------------------------------------------------------------------------------
# 5. Launch
# ------------------------------------------------------------------------------
if __name__ == "__main__":
    # In Colab, inline launch is preferred.
    # share=True provides a public URL for easy testing.
    demo.launch(debug=True, share=True, inline=True)

WARNING    /tmp/ipython-input-1854076160.py:312: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=css) as demo:
 [py.warnings]
  with gr.Blocks(theme=gr.themes.Soft(), css=css) as demo:



Failed to load AI pipeline: Unrecognized model in prajjwal1/bert-tiny. Should have a `model_type` key in its config.json.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9fdc4289ca7aad6b07.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7864 <> https://9fdc4289ca7aad6b07.gradio.live
